# CarSathi AI — 06 Model Evaluation & Tuning

Model comparison, error analysis, Random Forest tuning, cross-validation, final validation, residual analysis, and feature importance.


## Prerequisites / Setup


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_PATH = Path(
    r"C:\Users\lenovo\Document_local\Data Science NareshIT Course"
    r"\Projects For GITHUB\CarSathi-AI\data\processed\cars_ml.csv"
)

cars_ml = pd.read_csv(PROCESSED_PATH)

# Recreate the feature used by the original notebook.
CURRENT_YEAR = 2025
cars_ml["car_age"] = CURRENT_YEAR - cars_ml["year"]

print("Processed ML dataset loaded.")
print("Shape:", cars_ml.shape)
print("Columns:", cars_ml.columns.tolist())

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

numerical_features = [
    "km_driven", "mileage", "engine", "max_power",
    "seats", "torque_nm", "car_age"
]
categorical_features = [
    "fuel", "seller_type", "transmission", "owner"
]
features = numerical_features + categorical_features

X = cars_ml[features]
y = cars_ml["selling_price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_processed, y_train)

print("Random Forest model prepared for evaluation.")


In [ ]:
# Model Comparison & Feature Importance

In [ ]:
# Get feature importance

In [ ]:
feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print(feature_importance)

In [ ]:
# Show top 15 features

In [ ]:
print(
    feature_importance.head(15).to_string(index=False)
)

In [ ]:
# Visualize feature importance

In [ ]:
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"][::-1],
    top_features["Importance"][::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")

plt.show()

In [ ]:
# Model Comparison & Error Analysis

In [ ]:
# ============================================
# STEP 20: RANDOM FOREST ERROR ANALYSIS
# ============================================

# Predictions
y_pred_rf = rf_model.predict(X_test_processed)

# Create error analysis dataframe
error_df = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": y_pred_rf
})

# Calculate errors
error_df["Error"] = error_df["Actual_Price"] - error_df["Predicted_Price"]
error_df["Absolute_Error"] = error_df["Error"].abs()
error_df["Percentage_Error"] = (
    error_df["Absolute_Error"] / error_df["Actual_Price"]
) * 100

print("=" * 60)
print("RANDOM FOREST ERROR ANALYSIS")
print("=" * 60)

print("\nAverage Absolute Error:")
print(error_df["Absolute_Error"].mean())

print("\nMedian Absolute Error:")
print(error_df["Absolute_Error"].median())

print("\nMaximum Absolute Error:")
print(error_df["Absolute_Error"].max())

print("\nMean Percentage Error:")
print(error_df["Percentage_Error"].mean())

print("\nMedian Percentage Error:")
print(error_df["Percentage_Error"].median())

In [ ]:
print("\nTop 10 Worst Predictions:")

print(
    error_df
    .sort_values("Absolute_Error", ascending=False)
    .head(10)
    .to_string(index=False)
)

In [ ]:
# Random Forest Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# ============================================
# STEP 21: RANDOM FOREST HYPERPARAMETER TUNING
# ============================================

rf_base = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [None, 15, 25],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 1.0]
}

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_processed, y_train)

print("\n" + "=" * 60)
print("BEST RANDOM FOREST PARAMETERS")
print("=" * 60)

print(grid_search.best_params_)

print("\nBest Cross-Validation R²:")
print(grid_search.best_score_)

In [ ]:
# Evalute the models 

In [ ]:

best_rf = grid_search.best_estimator_

y_pred_tuned = best_rf.predict(X_test_processed)

tuned_mae = mean_absolute_error(y_test, y_pred_tuned)
tuned_rmse = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
tuned_r2 = r2_score(y_test, y_pred_tuned)

print("\n" + "=" * 60)
print("TUNED RANDOM FOREST RESULTS")
print("=" * 60)

print(f"MAE  : ₹{tuned_mae:,.2f}")
print(f"RMSE : ₹{tuned_rmse:,.2f}")
print(f"R²   : {tuned_r2:.4f}")

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "MAE": [
        166385.408809,
        100841.493428,
        78453.696461,
        tuned_mae
    ],
    "RMSE": [
        312311.420735,
        187423.952860,
        148550.800860,
        tuned_rmse
    ],
    "R2": [
        0.652736,
        0.874935,
        0.921434,
        tuned_r2
    ]
})

print(comparison.to_string(index=False))

In [ ]:
# Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

# ============================================
# STEP 22: 5-FOLD CROSS VALIDATION
# ============================================

cv_scores = cross_val_score(
    best_rf,
    X_train_processed,
    y_train,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("=" * 60)
print("5-FOLD CROSS VALIDATION")
print("=" * 60)

print("\nFold R² Scores:")
print(cv_scores)

print("\nMean CV R²:")
print(cv_scores.mean())

print("\nStandard Deviation:")
print(cv_scores.std())

print("\nMinimum R²:")
print(cv_scores.min())

print("\nMaximum R²:")
print(cv_scores.max())

In [ ]:
# Final Model Validation

In [ ]:
# ============================================
# STEP 23: FINAL MODEL VALIDATION
# ============================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Final predictions
final_pred = best_rf.predict(X_test_processed)

# Metrics
final_mae = mean_absolute_error(y_test, final_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, final_pred))
final_r2 = r2_score(y_test, final_pred)

print("=" * 60)
print("FINAL MODEL VALIDATION")
print("=" * 60)

print(f"\nMAE  : ₹{final_mae:,.2f}")
print(f"RMSE : ₹{final_rmse:,.2f}")
print(f"R²   : {final_r2:.4f}")

In [ ]:
# ============================================
# FINAL PREDICTION ERROR
# ============================================

final_error_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": final_pred
})

final_error_df["Absolute_Error"] = (
    final_error_df["Actual"] -
    final_error_df["Predicted"]
).abs()

final_error_df["Percentage_Error"] = (
    final_error_df["Absolute_Error"] /
    final_error_df["Actual"]
) * 100

print("\nMedian Absolute Error:")
print(
    final_error_df["Absolute_Error"].median()
)

print("\nMedian Percentage Error:")
print(
    final_error_df["Percentage_Error"].median()
)

In [ ]:
# Actual vs Predicted Analysis

In [ ]:
# Generate predictions using your tuned model first
tuned_rf_predictions = best_rf.predict(X_test_processed)  # Replace 'best_rf' with your model variable name

# ============================================
# STEP 21: ACTUAL vs PREDICTED ANALYSIS
# ============================================

comparison_df = pd.DataFrame({
    "Actual_Price": y_test.values,
    "Predicted_Price": tuned_rf_predictions
})

comparison_df["Error"] = (
    comparison_df["Actual_Price"]
    - comparison_df["Predicted_Price"]
)

comparison_df["Absolute_Error"] = (
    comparison_df["Error"].abs()
)

comparison_df["Percentage_Error"] = (
    comparison_df["Absolute_Error"]
    / comparison_df["Actual_Price"]
) * 100

print("=" * 60)
print("ACTUAL vs PREDICTED ANALYSIS")
print("=" * 60)

print("\nFirst 20 Predictions:")
print(comparison_df.head(20).to_string(index=False))

print("\nAverage Absolute Error:")
print(comparison_df["Absolute_Error"].mean())

print("\nMedian Absolute Error:")
print(comparison_df["Absolute_Error"].median())

print("\nMean Percentage Error:")
print(comparison_df["Percentage_Error"].mean())

print("\nMedian Percentage Error:")
print(comparison_df["Percentage_Error"].median())

print("\nBest 10 Predictions:")
print(
    comparison_df
    .sort_values("Absolute_Error")
    .head(10)
    .to_string(index=False)
)

print("\nWorst 10 Predictions:")
print(
    comparison_df
    .sort_values("Absolute_Error", ascending=False)
    .head(10)
    .to_string(index=False)
)

In [ ]:
# Residual Analysis


In [ ]:
# ============================================
# STEP 22.1: RESIDUAL STATISTICS
# ============================================

print("=" * 60)
print("RESIDUAL ANALYSIS")
print("=" * 60)

print("\nResidual Statistics:")

print(
    comparison_df["Error"].describe()
)

print("\nMean Residual:")
print(comparison_df["Error"].mean())

print("\nMedian Residual:")
print(comparison_df["Error"].median())

print("\nUnderpredictions:")
print(
    (comparison_df["Error"] > 0).sum()
)

print("\nOverpredictions:")
print(
    (comparison_df["Error"] < 0).sum()
)

print("\nAlmost Perfect Predictions:")
print(
    (comparison_df["Error"].abs() < 10000).sum()
)

In [ ]:
# Underprediction vs Overprediction

In [ ]:
# ============================================
# STEP 22.2: UNDER vs OVER PREDICTION
# ============================================

under_predictions = comparison_df[
    comparison_df["Error"] > 0
]

over_predictions = comparison_df[
    comparison_df["Error"] < 0
]

print("=" * 60)
print("UNDERPREDICTION vs OVERPREDICTION")
print("=" * 60)

print("\nUnderpredictions:")
print(
    len(under_predictions)
)

print("\nOverpredictions:")
print(
    len(over_predictions)
)

print("\nAverage Underprediction Error:")
print(
    under_predictions["Absolute_Error"].mean()
)

print("\nAverage Overprediction Error:")
print(
    over_predictions["Absolute_Error"].mean()
)

In [ ]:
# Error by Actual Price Range

In [ ]:
# ============================================
# STEP 22.3: ERROR BY PRICE RANGE
# ============================================

comparison_df["Price_Range"] = pd.cut(
    comparison_df["Actual_Price"],
    bins=[
        0,
        500000,
        1000000,
        2000000,
        5000000,
        float("inf")
    ],
    labels=[
        "Below ₹5L",
        "₹5L–₹10L",
        "₹10L–₹20L",
        "₹20L–₹50L",
        "Above ₹50L"
    ]
)

price_error_analysis = (
    comparison_df
    .groupby("Price_Range", observed=True)
    .agg(
        Count=("Actual_Price", "count"),
        MAE=("Absolute_Error", "mean"),
        Median_Error=("Absolute_Error", "median"),
        Mean_Percentage_Error=("Percentage_Error", "mean")
    )
    .reset_index()
)

print("=" * 60)
print("ERROR BY PRICE RANGE")
print("=" * 60)

print(
    price_error_analysis.to_string(index=False)
)

In [ ]:
# Residual vs Actual Price

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    comparison_df["Actual_Price"],
    comparison_df["Error"],
    alpha=0.5
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Actual Selling Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs Actual Selling Price")

plt.show()

In [ ]:
# Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    comparison_df["Actual_Price"],
    comparison_df["Predicted_Price"],
    alpha=0.5
)

min_price = min(
    comparison_df["Actual_Price"].min(),
    comparison_df["Predicted_Price"].min()
)

max_price = max(
    comparison_df["Actual_Price"].max(),
    comparison_df["Predicted_Price"].max()
)

plt.plot(
    [min_price, max_price],
    [min_price, max_price],
    linestyle="--"
)

plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title("Actual vs Predicted Car Selling Price")

plt.show()

In [ ]:
# Create Feature Importance DataFrame

In [ ]:
# Use the trained model variable directly
tuned_rf_model = rf_model

# STEP 23.1: FEATURE IMPORTANCE
feature_importance = pd.DataFrame({
    "Feature": preprocessor.get_feature_names_out(),
    "Importance": tuned_rf_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

print("=" * 60)
print("TUNED RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 60)

print(
    feature_importance
    .head(15)
    .to_string(index=False)
)

In [ ]:
# Calculate Percentage Importance

In [ ]:
feature_importance["Importance_Percentage"] = (
    feature_importance["Importance"] * 100
)

print(
    feature_importance
    .head(15)
    .to_string(index=False)
)

In [ ]:
# Visualize Top Features

In [ ]:
import matplotlib.pyplot as plt

top_features = feature_importance.head(10).copy()

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 10 Features — Tuned Random Forest")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Clean the Feature Names

In [ ]:
feature_importance["Clean_Feature"] = (
    feature_importance["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

print(
    feature_importance[
        ["Clean_Feature", "Importance_Percentage"]
    ]
    .head(15)
    .to_string(index=False)
)

In [ ]:
# Professional Feature Importance Chart

In [ ]:
top_features = feature_importance.head(10).copy()

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Clean_Feature"],
    top_features["Importance_Percentage"]
)

plt.xlabel("Importance (%)")
plt.ylabel("Feature")
plt.title("Top 10 Features Influencing Car Price Prediction")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance

# Run permutation importance on processed test data
perm_result = permutation_importance(
    tuned_rf_model,
    X_test_processed,  # Use transformed numeric features
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="r2"
)

# Extract feature names from preprocessor
feature_names = preprocessor.get_feature_names_out()

permutation_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_STD": perm_result.importances_std
}).sort_values("Importance_Mean", ascending=False)

print("=" * 60)
print("PERMUTATION FEATURE IMPORTANCE")
print("=" * 60)
print(permutation_importance_df.head(15).to_string(index=False))

In [ ]:
# Final Model Comparison & Selection

In [ ]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "MAE": [
        166385.408809,
        100841.493428,
        78453.696461,
        77856.890492
    ],
    "RMSE": [
        312311.420735,
        187423.952860,
        148550.800860,
        146378.377123
    ],
    "R2": [
        0.652736,
        0.874935,
        0.921434,
        0.923715
    ]
})

final_model_comparison

In [ ]:
final_model_comparison.sort_values(
    by="R2",
    ascending=False
)

In [ ]:
# Calculate improvement

In [ ]:
lr_mae = 166385.408809
rf_mae = 77856.890492

mae_improvement = ((lr_mae - rf_mae) / lr_mae) * 100

print(f"MAE Improvement: {mae_improvement:.2f}%")

In [ ]:
lr_r2 = 0.652736
rf_r2 = 0.923715

r2_improvement = ((rf_r2 - lr_r2) / lr_r2) * 100

print(f"R² Improvement: {r2_improvement:.2f}%")

In [ ]:
#Check Test vs Cross-Validation

In [ ]:
test_r2 = 0.923715
cv_r2 = 0.8980168716496622

gap = test_r2 - cv_r2

print(f"Test-CV R² Gap: {gap:.4f}")

In [ ]:
# conclusion

# It is Best From Result
# Tuned Random Forest

# MAE  = ₹77,856.89
# RMSE = ₹146,378.38
# R²   = 0.9237

# 5-Fold Validation
# Mean CV R² = 0.8980
# Std        = 0.0239